# 06 - Export Analytics-Ready Dataset (Power BI)

**Input:** `data/processed/train_feat.csv` | `val_feat.csv` | `test_feat.csv` (DVC-tracked)

**Output:** `data/analytics/housing_analytics_ready.csv`

---

### What this notebook does

| Step | Action |
|---|---|
| 1 | Merge train + val + test splits into one full dataset |
| 2 | Keep only business-readable features (no Scaling, no Encoding) |
| 3 | Add human-readable labels for categorical columns |
| 4 | Save to `data/analytics/housing_analytics_ready.csv` |

### Why a separate notebook?

The feature-engineered splits (`*_feat.csv`) are split for **model training**.
Power BI needs a **single flat file** with business-friendly columns — no StandardScaler output, no one-hot encoding.

> This notebook is **read-only** from the model pipeline perspective.
> It does not modify any DVC-tracked file.

---

## 0 - Setup

In [ ]:
import os
import sys
from pathlib import Path

repo_path = Path("/content/california_housing_full_project")
os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print(f"✅ Working dir : {os.getcwd()}")

## 1 - Imports

In [ ]:
import pandas as pd

print("Imports ready")

## 2 - Load Feature-Engineered Splits

In [ ]:
train = pd.read_csv("data/processed/train_feat.csv")
val   = pd.read_csv("data/processed/val_feat.csv")
test  = pd.read_csv("data/processed/test_feat.csv")

print(f"train : {train.shape}")
print(f"val   : {val.shape}")
print(f"test  : {test.shape}")

## 3 - Merge All Splits

> Power BI doesn't care about train/val/test — it needs the full picture.
> We add a `split` column so you can still filter by it in the dashboard if needed.

In [ ]:
train["split"] = "train"
val["split"]   = "val"
test["split"]  = "test"

df = pd.concat([train, val, test], ignore_index=True)

print(f"Full dataset : {df.shape[0]:,} rows x {df.shape[1]} cols")
print()
print("Columns:")
print(df.columns.tolist())

## 4 - Select Analytics Columns

We keep **business features only** — columns a stakeholder can understand without knowing the model.

| Keep | Reason |
|---|---|
| `longitude`, `latitude` | Map visualization in Power BI |
| `housing_median_age` | Age analysis |
| `median_income` | Income distribution |
| `median_house_value` | Target variable — the KPI |
| `ocean_proximity` | Geographic category |
| `rooms_per_household` | Engineered — space per family |
| `bedrooms_per_room` | Engineered — bedroom ratio |
| `population_per_household` | Engineered — density |
| `dist_SF`, `dist_LA` | Engineered — distance to hubs |
| `is_capped` | Flag — value hit the $500k ceiling |
| `split` | Which split this row came from |

❌ **NOT included:** StandardScaler output, one-hot encoded columns — those are model-only.

In [ ]:
ANALYTICS_COLS = [
    # Geographic
    "longitude",
    "latitude",
    "ocean_proximity",
    "dist_SF",
    "dist_LA",
    # Property
    "housing_median_age",
    "rooms_per_household",
    "bedrooms_per_room",
    "population_per_household",
    # Economic
    "median_income",
    "median_house_value",
    "is_capped",
    # Metadata
    "split",
]

# Keep only columns that exist (safe if some are missing)
available = [c for c in ANALYTICS_COLS if c in df.columns]
missing   = [c for c in ANALYTICS_COLS if c not in df.columns]

if missing:
    print(f"⚠️  Missing columns (skipped): {missing}")

df_pbi = df[available].copy()

print(f"Analytics dataset : {df_pbi.shape[0]:,} rows x {df_pbi.shape[1]} cols")
print()
print(df_pbi.dtypes)

## 5 - Add Human-Readable Labels

Power BI can filter on these directly — no need to remember numeric codes.

In [ ]:
# is_capped: 0/1 → readable label
if "is_capped" in df_pbi.columns:
    df_pbi["price_capped"] = df_pbi["is_capped"].map({1: "Capped at $500k", 0: "Normal"})

# income_tier: bin median_income into readable brackets
if "median_income" in df_pbi.columns:
    df_pbi["income_tier"] = pd.cut(
        df_pbi["median_income"],
        bins=[0, 2, 4, 6, 8, 100],
        labels=["Very Low", "Low", "Medium", "High", "Very High"],
    )

# age_group: bin housing age
if "housing_median_age" in df_pbi.columns:
    df_pbi["age_group"] = pd.cut(
        df_pbi["housing_median_age"],
        bins=[0, 10, 20, 30, 40, 100],
        labels=["<10 yrs", "10-20 yrs", "20-30 yrs", "30-40 yrs", "40+ yrs"],
    )

print("Labels added:")
print(df_pbi[["is_capped", "price_capped"]].value_counts() if "is_capped" in df_pbi.columns else "")
print()
print(df_pbi["income_tier"].value_counts() if "income_tier" in df_pbi.columns else "")

## 6 - Final Checks

In [ ]:
print("-- Shape --")
print(f"  {df_pbi.shape[0]:,} rows x {df_pbi.shape[1]} cols")

print()
print("-- Nulls --")
null_counts = df_pbi.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "  No nulls ✅")

print()
print("-- Sample rows --")
print(df_pbi.head(3).to_string())

## 7 - Save Analytics Dataset

In [ ]:
output_dir = Path("data/analytics")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "housing_analytics_ready.csv"
df_pbi.to_csv(output_path, index=False)

size_mb = output_path.stat().st_size / 1024**2
print(f"✅ Saved: {output_path}")
print(f"   Size : {size_mb:.2f} MB")
print(f"   Rows : {df_pbi.shape[0]:,}")
print(f"   Cols : {df_pbi.shape[1]}")

## 8 - Git Commit

> ⚠️ `data/analytics/` is **NOT DVC-tracked** — it's a derived export file.
> We commit it directly to Git so Power BI / teammates can download it easily.

```bash
git add data/analytics/housing_analytics_ready.csv
git add notebooks/06_export_powerbi.ipynb
git commit -m "analytics: add Power BI ready dataset"
git push
```

## Summary

| Column | Type | Use in Power BI |
|---|---|---|
| `longitude`, `latitude` | float | Map visual |
| `ocean_proximity` | category | Slicer / filter |
| `dist_SF`, `dist_LA` | float | Distance scatter |
| `housing_median_age` | float | Age distribution |
| `age_group` | category | Bar chart |
| `rooms_per_household` | float | Space analysis |
| `bedrooms_per_room` | float | Bedroom ratio |
| `population_per_household` | float | Density analysis |
| `median_income` | float | Income analysis |
| `income_tier` | category | Income slicer |
| `median_house_value` | float | **Main KPI** |
| `price_capped` | category | Capped filter |
| `split` | category | Data audit |

**Next step:** Open Power BI Desktop → Get Data → CSV → `data/analytics/housing_analytics_ready.csv`